# Solar Wind Dataset EDA

This notebook explores the `juliensimon/solar-wind` dataset for the geomagnetic storm prediction project.

The goal of this EDA is to:
- understand what the dataset represents
- inspect the columns and data types
- check missing values
- identify possible outliers
- create basic visualizations for useful variables
- note potential features for predicting geomagnetic storms

## Dataset Background

This dataset contains real-time solar wind plasma and magnetic field measurements from the DSCOVR and ACE spacecraft at the L1 Lagrange point. The measurements are collected upstream of Earth, meaning they can provide short lead-time information before solar wind reaches Earth's magnetosphere. 

The dataset is relevant to geomagnetic storm prediction because solar wind speed, density, and magnetic field orientation are major drivers of geomagnetic activitiy. In particular, the `bz_gsm` variable is important because sustained negative Bz indicates a southward interplanetary magnetic field, which can couple with Earth's magnetosphere and trigger geomagnetic storms. 

This dataset may help connect solar activity to geomagnetic storm outcomes in the broader causal chain:

solar flare -> CME -> solar wind -> geomagnetic indices/storms

### Plain-english Initial Understanding

The dataset is the solar wind dataset. It contains around one-minute measurements of solar wind plasma and magnetic field conditions from spacecraft located at the L1 point, between the Sun and Earth. This is useful because it captures the solar wind shortly before it reaches Earth's magnetosphere. The key variables include density, speed, temperature, total magnetic field strength, and magnetic field components in GSM coordinates. Among these, Bz is especially important because sustained negative Bz can allow stronger coupling with Earth's magnetic field and contribute to geomagnetic storms. 

**Important Note: dataset is updated daily, so row count may differ from the dataset card preview.**

## Research Notes / Terms to Understand

Because our group is still building background knowledge in space weather, this EDA will document unfamiliar variables, units, and scientific concepts as we inspect the dataset. 

Terms that may need further research:
- solar wind
- DSCOVR
- ACE
- L1 Lagrange point
- IMF
- GSM coordinates
- Bx, By, Bz
- southward Bz
- magnetic reconnection
- Kp index
- Dst index

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# install hugging face datasets if neededz
%pip install datasets -q
from datasets import load_dataset


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
dataset = load_dataset("juliensimon/solar-wind")
dataset

README.md: 0.00B [00:00, ?B/s]

data/solar_wind.parquet:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/123316 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['time_tag', 'density', 'speed', 'temperature', 'bt', 'bx_gsm', 'by_gsm', 'bz_gsm'],
        num_rows: 123316
    })
})

In [5]:
ds = dataset["train"]
df = ds.to_pandas()

df.head()

,time_tag,density,speed,temperature,bt,bx_gsm,by_gsm,bz_gsm
0,2026-03-17 08:31:00,0.38,467.8,57895.0,4.49,-3.47,2.74,-0.79
1,2026-03-17 08:32:00,0.41,471.6,71499.0,4.58,-3.38,3.08,-0.21
2,2026-03-17 08:33:00,0.42,472.1,76974.0,4.39,-3.21,2.99,-0.10
3,2026-03-17 08:34:00,0.45,465.1,51641.0,4.27,-3.45,2.47,-0.47
4,2026-03-17 08:35:00,0.95,489.6,103878.0,4.39,-3.49,2.63,-0.33


In [6]:
df.shape

(123316, 8)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123316 entries, 0 to 123315
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   time_tag     123316 non-null  datetime64[ns]
 1   density      119945 non-null  float64       
 2   speed        119443 non-null  float64       
 3   temperature  118866 non-null  float64       
 4   bt           120882 non-null  float64       
 5   bx_gsm       120882 non-null  float64       
 6   by_gsm       120882 non-null  float64       
 7   bz_gsm       120882 non-null  float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 7.5 MB


## Column Dictionary

| Column | Type | Meaning | Unit / Notes | Initial modeling relevance |
|---|---|---|---|---|
| `time_tag` | datetime | Measurement timestamp from DSCOVR/ACE at the L1 Lagrange point | UTC, approximately 1-minute cadence | Needed for time-based merging with storm labels or geomagnetic indices |
| `density` | numerical | Solar wind proton number density | protons/cm³ | Higher density can increase solar wind dynamic pressure on Earth's magnetosphere |
| `speed` | numerical | Solar wind bulk velocity | km/s | High solar wind speed can be associated with stronger geomagnetic activity |
| `temperature` | numerical | Solar wind proton temperature | Kelvin | May help identify solar wind structures such as magnetic clouds or CME-related conditions |
| `bt` | numerical | Total interplanetary magnetic field magnitude | nT | Stronger magnetic field can indicate more geoeffective solar wind conditions |
| `bx_gsm` | numerical | IMF Bx component in GSM coordinates | nT | Magnetic field component in the Sun-Earth direction |
| `by_gsm` | numerical | IMF By component in GSM coordinates | nT | May relate to asymmetric magnetospheric convection/current systems |
| `bz_gsm` | numerical | IMF Bz component in GSM coordinates | nT | Key storm driver; sustained negative/southward Bz can couple with Earth's magnetosphere |

In [8]:
# column summary table
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean() * 100).round(2).values,
    "unique_count": df.nunique().values
})

column_summary

,column,dtype,non_null_count,missing_count,missing_percent,unique_count
0,time_tag,datetime64[ns],123316,0,0.00,123316
1,density,float64,119945,3371,2.73,2509
2,speed,float64,119443,3873,3.14,4738
3,temperature,float64,118866,4450,3.61,90103
4,bt,float64,120882,2434,1.97,2621
5,bx_gsm,float64,120882,2434,1.97,3013
6,by_gsm,float64,120882,2434,1.97,3746
7,bz_gsm,float64,120882,2434,1.97,3123


## Initial Observations

- The dataset has one datetime column (`time_tag`) and seven numerical columns.
- `time_tag` has no missing values, which is important because this dataset will likely need to be merged with storm lables or geomagnetic indices by timestamp.
- All solar wind measurement columns have some missing values.
- Magnetic field columns (`bt`, `bx_gsm`, `by_gsm`, `bz_gsm`) appear to have the same non-null count, suggesting they may come from the same instrument/source or be missing together. 
- Plasma varaibles (`density`, `speed`, `temperature`) have slightly different missingness patterns.
- Since the dataset is updated daily, row count may differ from the dataset card preview.

## Plain-English Understanding

This dataset represents solar wind conditions measured shortly before the solar wind reaches Earth. The spacecraft are located at the L1 point between the Sun and Earth, so the dataset can provide short lead-time information for geomagnetic storm prediction.

In simple terms, the variables describe:
- how dense the solar wind is (`density`)
- how fast it is moving (`speed`)
- how hot the plasma is (`temperature`)
- how strong the magnetic field is overall (`bt`)
- which direction the magnetic field points (`bx_gsm`, `by_gsm`, `bz_gsm`)

The most important variable appears to be `bz_gsm`. When Bz is negative, the solar wind magnetic field points southward. This can connect more strongly with Earth's magnetic field and transfer energy into Earth's magnetosphere, which can contribute to geomagnetic storms. 
